# CSI2PointCloud — ENTRENAMIENTO

Entrena el modelo con el **dataset propio** (ESP32 + Kinect, 6 clases) en formato MM-Fi.
Usa el **modelo de Määttä** y el **dataset** incluidos en este repo.

- Split **70/15/15 cronológico por acción** (sin fuga temporal).
- Guarda el mejor modelo por validación en `resultados/modelo/csi2pc_v2.pth` y el `split.json`
  (que luego usa **test.ipynb** para evaluar exactamente el mismo test).

> Ejecuta en orden (Run All). En CPU tarda; con GPU es mucho más rápido.

## 1. Configuración y rutas

In [ ]:
import os, sys, glob, time, json
import numpy as np, scipy.io as scio, torch
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt

HERE = os.path.abspath('')
REPO = os.environ.get('CSI2PC_REPO', os.path.join(HERE, '..', 'csi2pointcloud'))
DATA = os.environ.get('CSI2PC_DATA', os.path.join(HERE, '..', 'mmfi_dataset'))
OUT  = os.environ.get('CSI2PC_OUT',  os.path.join(HERE, 'resultados'))
os.makedirs(os.path.join(OUT, 'modelo'), exist_ok=True)
sys.path.append(REPO)
from src.csi2pointcloud.csi2pointcloud_arch01 import CSI2PointCloudModel
from src.loss.mmslab_loss import CSI2PointCloudLoss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Modelo:', REPO); print('Datos :', DATA); print('Device:', device)

## 2. Dataset y split 70/15/15 cronológico por acción

In [ ]:
class CSIDataset(Dataset):
    def __init__(self, root):
        self.items, self.actions = [], []
        for mat in sorted(glob.glob(os.path.join(root, '**', 'wifi-csi', '*.mat'), recursive=True)):
            b = mat.replace('wifi-csi', 'lidar').replace('.mat', '.bin')
            if os.path.exists(b):
                self.items.append((mat, b))
                self.actions.append(os.path.basename(os.path.dirname(os.path.dirname(mat))))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mat, binf = self.items[i]
        m = scio.loadmat(mat)
        amp = m['CSIamp'].astype(np.float64); pha = m['CSIphase'].astype(np.float64)
        amp[np.isinf(amp)] = np.nan; pha[np.isinf(pha)] = np.nan
        if np.any(np.isnan(amp)): amp[np.isnan(amp)] = np.nanmean(amp)
        if np.any(np.isnan(pha)): pha[np.isnan(pha)] = np.nanmean(pha)
        amp = (amp - np.nanmin(amp)) / (np.nanmax(amp) - np.nanmin(amp) + 1e-9)
        csi = np.stack([amp, pha], axis=2).astype(np.float32)
        x = torch.from_numpy(csi); x = (x - x.mean()) / (x.std() + 1e-9)
        gt = np.frombuffer(open(binf, 'rb').read(), dtype=np.float64).reshape(-1, 3).astype(np.float32)
        g = torch.from_numpy(gt.copy()); g = (g - g.mean()) / (g.std() + 1e-9)
        return x, g

def split_por_accion(ds, ratios=(0.70, 0.15, 0.15)):
    tr, va, te = [], [], []
    acts = np.array(ds.actions)
    for a in sorted(set(ds.actions)):
        idx = np.where(acts == a)[0]
        n = len(idx); n_tr = int(ratios[0]*n); n_va = int(ratios[1]*n)
        tr += idx[:n_tr].tolist(); va += idx[n_tr:n_tr+n_va].tolist(); te += idx[n_tr+n_va:].tolist()
    return tr, va, te

ds = CSIDataset(DATA)
tr, va, te = split_por_accion(ds)
json.dump({'train': tr, 'val': va, 'test': te, 'acciones': sorted(set(ds.actions))},
          open(os.path.join(OUT, 'split.json'), 'w'))
print('Frames:', len(ds), '| train', len(tr), '| val', len(va), '| test', len(te))
print('Acciones:', sorted(set(ds.actions)))
print('split.json guardado -> lo usara test.ipynb')

## 3. Entrenamiento
Ajusta `EPOCHS`. Guarda el mejor modelo por validación.

In [ ]:
EPOCHS = 60
BATCH  = 8
torch.manual_seed(0)

train_dl = DataLoader(Subset(ds, tr), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(Subset(ds, va), batch_size=BATCH)

model = CSI2PointCloudModel(256, 4, 3, 3, 1200, 3, 114, 10).to(device)
loss_fn = CSI2PointCloudLoss()
opt = torch.optim.NAdam(model.parameters(), lr=1e-4)

hist_tr, hist_val, best = [], [], None
for ep in range(EPOCHS):
    t0 = time.time(); model.train(); tl = []
    for x, g in train_dl:
        x, g = x.to(device), g.to(device)
        opt.zero_grad(); pred, tf = model(x); loss = loss_fn(pred, g, tf)
        loss.backward(); opt.step(); tl.append(loss.item())
    model.eval(); vl = []
    with torch.no_grad():
        for x, g in val_dl:
            x, g = x.to(device), g.to(device)
            pred, tf = model(x); vl.append(loss_fn(pred, g, tf).item())
    hist_tr.append(np.mean(tl)); hist_val.append(np.mean(vl) if vl else np.nan)
    print('Epoca %3d/%d | train %.1f | val %.1f | %.1fs' % (ep+1, EPOCHS, hist_tr[-1], hist_val[-1], time.time()-t0))
    if best is None or hist_val[-1] < best:
        best = hist_val[-1]
        torch.save({'model_state_dict': model.state_dict(), 'epochs': ep+1}, os.path.join(OUT, 'modelo', 'csi2pc_v2.pth'))
print('LISTO. Mejor val =', round(best, 2), '-> modelo/csi2pc_v2.pth')

## 4. Curva de pérdida

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(hist_tr, label='train'); plt.plot(hist_val, label='val')
plt.xlabel('epoca'); plt.ylabel('perdida (Chamfer + reg)'); plt.legend(); plt.grid(True)
plt.title('Entrenamiento CSI2PointCloud (6 clases)')
plt.savefig(os.path.join(OUT, 'curva_perdida.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Ahora abre  test.ipynb  para las metricas ICP y la visualizacion.')